# Original STATE downloads and complete primary-axis history

This is the source entry point for the 19 primary analysis notebooks. The original run downloaded official STATE scored outputs, small Tahoe metadata tables, training/validation split files and annotation snapshots. It did not run fresh STATE inference. This notebook shows that actual pipeline and preserves preliminary screens, including candidates discarded before a standalone curve was justified.

**Portable input setup.** Install `../requirements.txt` in a Python3.11 environment. The first code cell restores the exact offline payloads included in this checkout, verifies their hashes, and then runs the original scored-output analysis. No model inference is performed. Historical saved curves remain visible; a clean run rebuilds its own `computed/` tables. See `../README.md`.

In [1]:
from pathlib import Path
import sys, json
root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'recovered_pipeline.py').is_file())
sys.path.insert(0,str(root))
import pandas as pd
from IPython.display import display, Markdown
from state_inputs import ensure_inputs
print(ensure_inputs())
import recovered_pipeline as rp
import notebook_support as ns


{'profile': 'default', 'verified_files': 226, 'assets': [{'asset': 'state-historical-comparison-tables.tar.gz', 'members': 25, 'source': 'verified_existing_inputs'}, {'asset': 'state-official-score-cache.tar.gz', 'members': 193, 'source': 'verified_existing_inputs'}, {'asset': 'state-historical-annotations.tar.gz', 'members': 8, 'source': 'verified_existing_inputs'}], 'new_model_inference': False}


Matplotlib is building the font cache; this may take a moment.


## Original download and processing implementation

In [2]:
initial,base=rp.modules()
import inspect
for f in [initial.get_hf_file_manifest,initial.load_target_results,initial.load_tahoe_metadata]:
    display(Markdown('```python\n'+inspect.getsource(f)+'\n```'))

/opt/homebrew/anaconda3/envs/fleming/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


```python
def get_hf_file_manifest():
    manifest_path = ART / "hf_resource_manifest.json"
    if manifest_path.exists():
        return json.loads(manifest_path.read_text())
    api = HfApi()
    files = []
    summary = []
    repos = [(r, "model") for r in MODEL_REPOS] + [
        ("arcinstitute/Replogle-Nadig-Preprint", "dataset"),
        ("arcinstitute/State-Parse-Filtered", "dataset"),
        ("tahoebio/Tahoe-100M", "dataset"),
        ("arcinstitute/SE-600M", "model"),
        ("arcinstitute/SE-100M", "model"),
    ]
    for repo, typ in repos:
        rec = {"repo_id": repo, "repo_type": typ, "status": "ok", "total_bytes": 0, "n_files": 0}
        try:
            info = api.model_info(repo, files_metadata=True) if typ == "model" else api.dataset_info(repo, files_metadata=True)
            for sib in info.siblings or []:
                size = getattr(sib, "size", None) or 0
                rec["total_bytes"] += size
                rec["n_files"] += 1
                files.append({"repo_id": repo, "repo_type": typ, "path": sib.rfilename, "size": size})
        except Exception as e:
            rec.update({"status": "error", "error": f"{type(e).__name__}: {str(e)[:300]}"})
        summary.append(rec)
    obj = {"summary": summary, "files": files}
    manifest_path.write_text(json.dumps(obj, indent=2))
    return obj

```

```python
def load_target_results(manifest):
    cache_dir = SPECTRA / "loaded_target_results"
    cache_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    download_log = []
    for f in result_file_rows(manifest):
        repo = f["repo_id"]
        path = f["path"]
        local = hf_hub_download(repo_id=repo, filename=path, cache_dir=str(CACHE))
        df = pd.read_csv(local)
        dataset, protocol, context, split_id, rep = parse_result_path(repo, path)
        df.insert(0, "source_repo", repo)
        df.insert(1, "source_path", path)
        df.insert(2, "dataset_family", dataset)
        df.insert(3, "protocol", protocol)
        df.insert(4, "context", context)
        df.insert(5, "split_id", split_id)
        df.insert(6, "representation", rep)
        names, doses, units = zip(*[parse_perturbation(x) for x in df["perturbation"].tolist()])
        df["perturbation_name"] = names
        df["dose"] = doses
        df["dose_unit"] = units
        df["perturbation_norm"] = [norm_name(x) for x in names]
        df["perturbation_loose"] = [loose_name(x) for x in names]
        frames.append(df)
        download_log.append({"repo": repo, "path": path, "rows": int(len(df)), "bytes": f.get("size", 0), "local": local})
    all_df = pd.concat(frames, ignore_index=True)
    all_df.to_csv(ROOT / "target_model_results.csv", index=False)
    (ART / "loaded_target_results_manifest.json").write_text(json.dumps(download_log, indent=2))
    return all_df, download_log

```

```python
def load_tahoe_metadata():
    drug_path = hf_hub_download(
        repo_id="tahoebio/Tahoe-100M",
        repo_type="dataset",
        filename="metadata/drug_metadata.parquet",
        cache_dir=str(CACHE),
    )
    gene_path = hf_hub_download(
        repo_id="tahoebio/Tahoe-100M",
        repo_type="dataset",
        filename="metadata/gene_metadata.parquet",
        cache_dir=str(CACHE),
    )
    cell_path = hf_hub_download(
        repo_id="tahoebio/Tahoe-100M",
        repo_type="dataset",
        filename="metadata/cell_line_metadata.parquet",
        cache_dir=str(CACHE),
    )
    return pd.read_parquet(drug_path), pd.read_parquet(gene_path), pd.read_parquet(cell_path)

```

## Rebuild original inputs and all analysis stages

The first execution creates `computed/` from the recovered original files. Completed stages are cached. Change `force=False` to `force=True` to repeat a stage. No inference job or remote write is performed.

In [3]:
verification={}
for stage in rp.DEPENDENCIES:
    verification[stage]=rp.run_stage(stage, force=False)
    print(stage, 'completed')
display(verification)

initial completed
replogle completed
parse completed
controls completed
zeroshot completed
ontology completed
history completed


{'initial': {'score_files': 55,
  'score_rows': 39808,
  'chemical_rows': 7288,
  'drug_metadata_rows': 379,
  'comparison': {'rows': 39808,
   'differences': {'source_repo': 0,
    'source_path': 0,
    'perturbation': 0,
    'overlap_at_N': 0,
    'de_sig_genes_recall': 0,
    'pearson_delta': 0,
    'max_tanimoto_to_target_context_train_drug': 0,
    'chem_support_level': 0,
    'target_context_train_drug_count': 0},
   'all_match': True},
  'tie_note': {'nearest_reference_label_differences': 50,
   'all_are_equal_similarity_ties': True,
   'explanation': 'Original code iterates a Python set and chooses the first maximum. Tied nearest-drug labels can differ with hash order; all similarities, level assignments and evaluated outcomes match.'}},
 'replogle': {'comparison': {'rows': 8116,
   'differences': {'reactome_pathway_count': 0,
    'reactome_max_jaccard_to_train': 0,
    'reactome_pathway_train_support_min': 0,
    'reactome_pathway_train_support_mean': 0,
    'reactome_pathway_

## Exact original score and metadata downloads

In [4]:
downloads=pd.read_json(rp.COMPUTED/'initial/artifacts/download_revisions.json')
with pd.option_context('display.max_rows',100,'display.max_colwidth',120):
    display(downloads[['repo','path','revision']])

,repo,path,revision
0,arcinstitute/ST-HVG-Tahoe,fewshot/state_generalization_X_hvg/eval_best.ckpt/C32_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
1,arcinstitute/ST-HVG-Tahoe,fewshot/state_generalization_X_hvg/eval_best.ckpt/HOP62_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
2,arcinstitute/ST-HVG-Tahoe,fewshot/state_generalization_X_hvg/eval_best.ckpt/HepG2-C3A_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
3,arcinstitute/ST-HVG-Tahoe,fewshot/state_generalization_X_hvg/eval_best.ckpt/Hs 766T_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
4,arcinstitute/ST-HVG-Tahoe,fewshot/state_generalization_X_hvg/eval_best.ckpt/PANC-1_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
5,arcinstitute/ST-HVG-Tahoe,zeroshot/state_generalization_zeroshot_X_hvg/eval_best.ckpt/C32_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
6,arcinstitute/ST-HVG-Tahoe,zeroshot/state_generalization_zeroshot_X_hvg/eval_best.ckpt/HOP62_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
7,arcinstitute/ST-HVG-Tahoe,zeroshot/state_generalization_zeroshot_X_hvg/eval_best.ckpt/HepG2-C3A_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
8,arcinstitute/ST-HVG-Tahoe,zeroshot/state_generalization_zeroshot_X_hvg/eval_best.ckpt/Hs 766T_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7
9,arcinstitute/ST-HVG-Tahoe,zeroshot/state_generalization_zeroshot_X_hvg/eval_best.ckpt/PANC-1_results.csv,ca6b751972493f8448e3256d1340ae70ad43e1e7


## Original hypothesis and disposition ledger

In [5]:
ns.display_json(rp.INITIAL/'artifacts/hypothesis_ledger.json')
ns.display_json(rp.INITIAL/'axis_ledger.json')
ns.display_json(rp.CONTINUED/'spectra_loop_state.json')

```json
[
  {
    "id": "C1_context_generalization",
    "source": "State manuscript abstract/results and README",
    "claim": "STATE predicts perturbation effects across diverse contexts and improves context generalization on chemical, signaling, and genetic perturbation datasets.",
    "scientific_unit": "perturbation-by-cell-context response example",
    "paper_protocol": "few-shot context generalization: target contexts contribute part of perturbations to training; remaining perturbations are held out in that context",
    "spectra_risk": "The same perturbation may be present in other training contexts, so this is not global perturbation novelty."
  },
  {
    "id": "C2_embedding_transfer",
    "source": "State manuscript results and README",
    "claim": "SE-trained representations enable transfer across datasets and improve novel-context performance with less data.",
    "scientific_unit": "dataset/context-level perturbation response examples",
    "paper_protocol": "ST+SE compared with ST+HVG and other baselines on public/preprint splits",
    "spectra_risk": "Precomputed model-card results are author-produced; independent target rerun needs checkpoints plus H5ADs."
  },
  {
    "id": "C3_novel_context_no_perturbations",
    "source": "State manuscript abstract and zero-shot TOMLs",
    "claim": "STATE can identify strong perturbations in novel cellular contexts where no perturbations were observed during training.",
    "scientific_unit": "perturbation-by-heldout-cell-context response example",
    "paper_protocol": "zeroshot context holdout",
    "spectra_risk": "Only five Tahoe zero-shot contexts in scored artifacts; context-similarity axis is underpowered without downloading/running more data."
  }
]
```

```json
[
  {
    "axis_id": "A1_tahoe_target_context_chemical_support",
    "status": "valid_negative_or_weak",
    "frozen_before_full_target_scoring": true,
    "schema_probe_exposure": "One C32 ST-HVG result CSV was opened before this script for column inspection only; confirmation summaries use the full loaded Tahoe pool and are not selected by target error.",
    "unit": "Tahoe perturbation-by-cell-context-by-dose examples, analyzed separately for ST-HVG and ST-SE",
    "axis_definition": "For each held-out Tahoe few-shot target-context drug/dose, compute RDKit Morgan Tanimoto similarity to the nearest drug that is observed in that same target context's training subset. Split into five equal-frequency levels using the similarity feature only.",
    "direction": "L1_near has highest target-context training support; L5_far has lowest support.",
    "validity_checks": {
      "nontrivial_levels": 5,
      "min_rows_per_level_per_representation": 720,
      "similarity_decreases_when_ordered": true,
      "label_or_target_error_used_for_membership": false,
      "target_model_confidence_used_for_membership": false,
      "fixed_baseline": "Zero-response baseline for DE overlap/recall and DE-count MAE; stronger context/perturb-mean baselines require raw train labels or author baseline artifacts not present in small scored CSVs.",
      "mapping_rate_exact_or_loose": 1.0
    },
    "claim_boundary": "Valid only for target-context chemical support in the official Tahoe few-shot context-generalization setup. It is not global unseen-drug generalization because the same drugs may appear in other training contexts.",
    "primary_artifacts": [
      "spectra_ready_dataset/tahoe_fewshot_chemical_support_examples.csv",
      "artifacts/spc_tahoe_fewshot_target_context_chemical_support.csv",
      "artifacts/spc_tahoe_fewshot_target_context_chemical_support_trends.csv"
    ]
  },
  {
    "axis_id": "A2_tahoe_dose_protocol",
    "status": "exploratory_invalid_as_spc_similarity",
    "axis_definition": "Dose level of held-out Tahoe perturbation (0.05, 0.5, 5.0 uM).",
    "reason": "Dose is prospective, but it is not a decreasing train-test similarity axis in the official splits because all dose levels are represented in training through other perturbations.",
    "primary_artifacts": [
      "artifacts/exploratory_tahoe_dose_protocol_curve.csv"
    ]
  },
  {
    "axis_id": "A3_all_scored_axis_discovery",
    "status": "outcome_informed_discovery_only",
    "axis_definition": "Prospective metadata features mined after scored evidence: dataset family, representation, Tahoe MOA fields, dose.",
    "reason": "These hypotheses require a frozen confirmation panel before any paper-facing claim.",
    "primary_artifacts": [
      "artifacts/all_scored_axis_discovery.json"
    ]
  }
]
```

```json
{
  "session": "state_generalizability_continued_20260603",
  "target_model_results": "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/target_model_results.csv",
  "prior_session": "/ewsc/yektefai/spectra_runs/state_generalizability_20260602_single_controller",
  "axes": [
    {
      "axis_id": "STATE_TAHOE_CHEMICAL_SUPPORT_PRIOR_A1",
      "status": "valid_negative_prior",
      "finding": "Prior target-context chemical Tanimoto support axis was split-valid but did not show meaningful degradation.",
      "source": "/ewsc/yektefai/spectra_runs/state_generalizability_20260602_single_controller/artifacts/spc_tahoe_fewshot_target_context_chemical_support.csv"
    },
    {
      "axis_id": "STATE_REPLOGLE_REACTOME_TRAIN_SUPPORT_V1",
      "status": "valid",
      "finding": "Replogle Reactome pathway support tested with discovery contexts and held-out confirmation contexts.",
      "artifacts": [
        "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/replogle_reactome_axis_freeze.json",
        "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/spc_replogle_fewshot_reactome_support.csv",
        "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/spc_replogle_fewshot_reactome_support_trends.csv",
        "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/spc_replogle_reactome_support_bootstrap.csv",
        "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/auditor_decision_replogle_reactome_support.json"
      ]
    },
    {
      "axis_id": "STATE_PARSE_LIGAND_FAMILY_SUPPORT_V1",
      "status": "exploratory_tested",
      "finding": "Parse ligand-family training support tested as next prospective axis; not used as terminal claim unless it confirms.",
      "artifacts": [
        "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/parse_ligand_family_axis_freeze.json",
        "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/spc_parse_fewshot_ligand_family_support.csv",
        "/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603/artifacts/spc_parse_fewshot_ligand_family_support_trends.csv"
      ]
    }
  ],
  "next_action_policy": "If Replogle Reactome support is not valid, use all_scored_axis_discovery_continued to freeze the next prospective axis and confirm on fresh/held-out evidence.",
  "all_scored_discovery": {
    "status": "exploratory_only_outcome_informed",
    "lowest_groups": [
      {
        "feature": "chem_support_level",
        "value": "L3_mid",
        "n_rows": 1458,
        "mean_overlap_at_N": 0.5053482340101367,
        "mean_de_sig_genes_recall": 0.6874874964745544,
        "mean_pearson_delta": 0.9051800990740376
      },
      {
        "feature": "chem_support_level",
        "value": "L1_near",
        "n_rows": 1458,
        "mean_overlap_at_N": 0.5135398378804875,
        "mean_de_sig_genes_recall": 0.6883161185407319,
        "mean_pearson_delta": 0.9272107665103813
      },
      {
        "feature": "chem_support_level",
        "value": "L4_low_mid",
        "n_rows": 1457,
        "mean_overlap_at_N": 0.5237620289839947,
        "mean_de_sig_genes_recall": 0.6942043632481113,
        "mean_pearson_delta": 0.9171364551660808
      },
      {
        "feature": "chem_support_level",
        "value": "L5_far",
        "n_rows": 1458,
        "mean_overlap_at_N": 0.5269851011647784,
        "mean_de_sig_genes_recall": 0.6867936252249953,
        "mean_pearson_delta": 0.9251058690341902
      },
      {
        "feature": "chem_support_level",
        "value": "L2_high_mid",
        "n_rows": 1457,
        "mean_overlap_at_N": 0.5321821993808319,
        "mean_de_sig_genes_recall": 0.7043593052208518,
        "mean_pearson_delta": 0.9212630636166773
      },
      {
        "feature": "context",
        "value": "Plasmablast",
        "n_rows": 304,
        "mean_overlap_at_N": 0.03882593839820908,
        "mean_de_sig_genes_recall": 0.11178108540026328,
        "mean_pearson_delta": 0.1945505806870642
      },
      {
        "feature": "context",
        "value": "jurkat",
        "n_rows": 5242,
        "mean_overlap_at_N": 0.12785554199062482,
        "mean_de_sig_genes_recall": 0.15992636548968855,
        "mean_pearson_delta": 0.2648305204836496
      },
      {
        "feature": "context",
        "value": "rpe1",
        "n_rows": 5120,
        "mean_overlap_at_N": 0.14937050786434644,
        "mean_de_sig_genes_recall": 0.17001842582280036,
        "mean_pearson_delta": 0.323329899679311
      },
      {
        "feature": "context",
        "value": "k562",
        "n_rows": 4702,
        "mean_overlap_at_N": 0.1494900310658744,
        "mean_de_sig_genes_recall": 0.21223331052710986,
        "mean_pearson_delta": 0.2736896187956806
      },
      {
        "feature": "context",
        "value": "hepg2",
        "n_rows": 4570,
        "mean_overlap_at_N": 0.15923109773006708,
        "mean_de_sig_genes_recall": 0.19989513084554897,
        "mean_pearson_delta": 0.31862171842800124
      },
      {
        "feature": "context",
        "value": "B_Intermediate_Memory",
        "n_rows": 304,
        "mean_overlap_at_N": 0.24156285578964037,
        "mean_de_sig_genes_recall": 0.47722098705610894,
        "mean_pearson_delta": 0.6866483891774949
      },
      {
        "feature": "context",
        "value": "HepG2-C3A",
        "n_rows": 3744,
        "mean_overlap_at_N": 0.27911220643489726,
        "mean_de_sig_genes_recall": 0.6476776097652778,
        "mean_pearson_delta": 0.595009094046758
      },
      {
        "feature": "context",
        "value": "Hs 766T",
        "n_rows": 3744,
        "mean_overlap_at_N": 0.2835824530569768,
        "mean_de_sig_genes_recall": 0.7539871385410485,
        "mean_pearson_delta": 0.6043491289893943
      },
      {
        "feature": "context",
        "value": "B_Naive",
        "n_rows": 304,
        "mean_overlap_at_N": 0.30981725399621984,
        "mean_de_sig_genes_recall": 0.5306902682014134,
        "mean_pearson_delta": 0.7173132692022544
      },
      {
        "feature": "context",
        "value": "PANC-1",
        "n_rows": 3740,
        "mean_overlap_at_N": 0.3220846318423647,
        "mean_de_sig_genes_recall": 0.7775233155221678,
        "mean_pearson_delta": 0.6485185430570243
      },
      {
        "feature": "context",
        "value": "C32",
        "n_rows": 3744,
        "mean_overlap_at_N": 0.3516863868577496,
        "mean_de_sig_genes_recall": 0.7097195308538881,
        "mean_pearson_delta": 0.6979396258543408
      },
      {
        "feature": "context",
        "value": "CD4_Memory",
        "n_rows": 242,
        "mean_overlap_at_N": 0.36415433145608306,
        "mean_de_sig_genes_recall": 0.7284848306831696,
        "mean_pearson_delta": 0.777534784118006
      },
      {
        "feature": "context",
        "value": "HOP62",
        "n_rows": 3744,
        "mean_overlap_at_N": 0.3684680952839311,
        "mean_de_sig_genes_recall": 0.7805967990445474,
        "mean_pearson_delta": 0.693994955053344
      },
      {
        "feature": "context",
        "value": "CD14_Mono",
        "n_rows": 304,
        "mean_overlap_at_N": 0.5861049474240378,
        "mean_de_sig_genes_recall": 0.651762780935924,
        "mean_pearson_delta": 0.7707339172300539
      },
      {
        "feature": "dataset_family",
        "value": "Replogle-Nadig",
        "n_rows": 19634,
        "mean_overlap_at_N": 0.14595008628289993,
        "mean_de_sig_genes_recall": 0.18438780290171716,
        "mean_pearson_delta": 0.294727539753826
      },
      {
        "feature": "dataset_family",
        "value": "Parse-PBMC",
        "n_rows": 1458,
        "mean_overlap_at_N": 0.3057091158280087,
        "mean_de_sig_genes_recall": 0.4944710045932112,
        "mean_pearson_delta": 0.6230550406521297
      },
      {
        "feature": "dataset_family",
        "value": "Tahoe-100M",
        "n_rows": 18716,
        "mean_overlap_at_N": 0.32098652005591327,
        "mean_de_sig_genes_recall": 0.733891555719787,
        "mean_pearson_delta": 0.6479621505128765
      },
      {
        "feature": "drugmeta_moa-broad",
        "value": "unclear",
        "n_rows": 3760,
        "mean_overlap_at_N": 0.317423902620782,
        "mean_de_sig_genes_recall": 0.7362889536804292,
        "mean_pearson_delta": 0.6351610712246335
      },
      {
        "feature": "drugmeta_moa-broad",
        "value": "inhibitor/antagonist",
        "n_rows": 13626,
        "mean_overlap_at_N": 0.3202897464366965,
        "mean_de_sig_genes_recall": 0.7325944900790952,
        "mean_pearson_delta": 0.651682725135209
      },
      {
        "feature": "drugmeta_moa-broad",
        "value": "activator/agonist",
        "n_rows": 1330,
        "mean_overlap_at_N": 0.33819680493677196,
        "mean_de_sig_genes_recall": 0.7404025332296013,
        "mean_pearson_delta": 0.6460339612797116
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "DNA methyltransferase inhibitor",
        "n_rows": 170,
        "mean_overlap_at_N": 0.27616100745483946,
        "mean_de_sig_genes_recall": 0.7781210876532728,
        "mean_pearson_delta": 0.6794741724508212
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "EGFR/ERBB inhibitor",
        "n_rows": 570,
        "mean_overlap_at_N": 0.28494762144612445,
        "mean_de_sig_genes_recall": 0.7390496565563354,
        "mean_pearson_delta": 0.6502342591911816
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Other TK inhibitor",
        "n_rows": 660,
        "mean_overlap_at_N": 0.297224189320638,
        "mean_de_sig_genes_recall": 0.7195225424467704,
        "mean_pearson_delta": 0.5878044243210532
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Sonic inhibitor",
        "n_rows": 140,
        "mean_overlap_at_N": 0.3003832503856378,
        "mean_de_sig_genes_recall": 0.747302258409999,
        "mean_pearson_delta": 0.5802327273679632
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Other MAPK inhibitor",
        "n_rows": 510,
        "mean_overlap_at_N": 0.30071389129715925,
        "mean_de_sig_genes_recall": 0.7126912274261489,
        "mean_pearson_delta": 0.6484068100797195
      }
    ],
    "highest_groups": [
      {
        "feature": "drugmeta_moa-fine",
        "value": "DNA methyltransferase inhibitor",
        "n_rows": 170,
        "mean_overlap_at_N": 0.27616100745483946,
        "mean_de_sig_genes_recall": 0.7781210876532728,
        "mean_pearson_delta": 0.6794741724508212
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "EGFR/ERBB inhibitor",
        "n_rows": 570,
        "mean_overlap_at_N": 0.28494762144612445,
        "mean_de_sig_genes_recall": 0.7390496565563354,
        "mean_pearson_delta": 0.6502342591911816
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Other TK inhibitor",
        "n_rows": 660,
        "mean_overlap_at_N": 0.297224189320638,
        "mean_de_sig_genes_recall": 0.7195225424467704,
        "mean_pearson_delta": 0.5878044243210532
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Sonic inhibitor",
        "n_rows": 140,
        "mean_overlap_at_N": 0.3003832503856378,
        "mean_de_sig_genes_recall": 0.747302258409999,
        "mean_pearson_delta": 0.5802327273679632
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Other MAPK inhibitor",
        "n_rows": 510,
        "mean_overlap_at_N": 0.30071389129715925,
        "mean_de_sig_genes_recall": 0.7126912274261489,
        "mean_pearson_delta": 0.6484068100797195
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Adrenoceptor agonist",
        "n_rows": 380,
        "mean_overlap_at_N": 0.3017080075419724,
        "mean_de_sig_genes_recall": 0.7291255700023127,
        "mean_pearson_delta": 0.6306648391323458
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "GSK3 inhibitor",
        "n_rows": 180,
        "mean_overlap_at_N": 0.30232215419776703,
        "mean_de_sig_genes_recall": 0.6880740571429423,
        "mean_pearson_delta": 0.6294599278726511
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Multi-TK inhibitor",
        "n_rows": 460,
        "mean_overlap_at_N": 0.3045801241017002,
        "mean_de_sig_genes_recall": 0.7460812868877836,
        "mean_pearson_delta": 0.6472068401529332
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "PI3K/AKT inhibitor",
        "n_rows": 270,
        "mean_overlap_at_N": 0.3097948316660418,
        "mean_de_sig_genes_recall": 0.7378480142991641,
        "mean_pearson_delta": 0.6724556157434428
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "unclear",
        "n_rows": 9750,
        "mean_overlap_at_N": 0.3109168530328336,
        "mean_de_sig_genes_recall": 0.7322774364620941,
        "mean_pearson_delta": 0.6363217213987261
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "MEK inhibitor",
        "n_rows": 240,
        "mean_overlap_at_N": 0.31133864665039707,
        "mean_de_sig_genes_recall": 0.7709765014972626,
        "mean_pearson_delta": 0.7043437694742655
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "RAS inhibitor",
        "n_rows": 210,
        "mean_overlap_at_N": 0.3250641374984229,
        "mean_de_sig_genes_recall": 0.7264401423661525,
        "mean_pearson_delta": 0.6381481252788078
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Cyclooxygenase inhibitor",
        "n_rows": 680,
        "mean_overlap_at_N": 0.32621346414191765,
        "mean_de_sig_genes_recall": 0.7413250942039727,
        "mean_pearson_delta": 0.6664491406470702
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Androgen receptor antagonist",
        "n_rows": 190,
        "mean_overlap_at_N": 0.326783988858525,
        "mean_de_sig_genes_recall": 0.7731401007844302,
        "mean_pearson_delta": 0.6225235913114279
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "JAK/STAT inhibitor",
        "n_rows": 340,
        "mean_overlap_at_N": 0.33892766175156264,
        "mean_de_sig_genes_recall": 0.7270949281604864,
        "mean_pearson_delta": 0.6246561726733275
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "DNA synthesis/repair inhibitor",
        "n_rows": 1290,
        "mean_overlap_at_N": 0.3399195006501739,
        "mean_de_sig_genes_recall": 0.7424030516286367,
        "mean_pearson_delta": 0.6514408248178201
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Glucose transporter inhibitor",
        "n_rows": 220,
        "mean_overlap_at_N": 0.36091802247429944,
        "mean_de_sig_genes_recall": 0.7556975926222972,
        "mean_pearson_delta": 0.6952141849559914
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "MTOR inhibitor",
        "n_rows": 420,
        "mean_overlap_at_N": 0.3627186511143625,
        "mean_de_sig_genes_recall": 0.7277103197194202,
        "mean_pearson_delta": 0.700735258268902
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "RAF inhibitor",
        "n_rows": 176,
        "mean_overlap_at_N": 0.3629422096549461,
        "mean_de_sig_genes_recall": 0.6643990483944567,
        "mean_pearson_delta": 0.5797192862142269
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "CDK inhibitor",
        "n_rows": 300,
        "mean_overlap_at_N": 0.36453195823604145,
        "mean_de_sig_genes_recall": 0.7198124812678253,
        "mean_pearson_delta": 0.7272001217771321
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Glucocorticoid receptor agonist",
        "n_rows": 280,
        "mean_overlap_at_N": 0.36815421698819356,
        "mean_de_sig_genes_recall": 0.7532829678104692,
        "mean_pearson_delta": 0.7641397176551566
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Protein synthesis inhibitor",
        "n_rows": 330,
        "mean_overlap_at_N": 0.36873169542511536,
        "mean_de_sig_genes_recall": 0.7100548156496329,
        "mean_pearson_delta": 0.6719573722446732
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Retinoic receptor agonist",
        "n_rows": 220,
        "mean_overlap_at_N": 0.3786191287712921,
        "mean_de_sig_genes_recall": 0.7496367407090879,
        "mean_pearson_delta": 0.6714592429724606
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "HDAC inhibitor",
        "n_rows": 170,
        "mean_overlap_at_N": 0.39235579480251054,
        "mean_de_sig_genes_recall": 0.7964869117419634,
        "mean_pearson_delta": 0.6798293925143833
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Microtubule inhibitor",
        "n_rows": 390,
        "mean_overlap_at_N": 0.3943236764800518,
        "mean_de_sig_genes_recall": 0.7422259447123177,
        "mean_pearson_delta": 0.716141456594834
      },
      {
        "feature": "drugmeta_moa-fine",
        "value": "Proteasome inhibitor",
        "n_rows": 170,
        "mean_overlap_at_N": 0.40125576042347677,
        "mean_de_sig_genes_recall": 0.7215210327942178,
        "mean_pearson_delta": 0.7757355345303522
      },
      {
        "feature": "protocol",
        "value": "zeroshot",
        "n_rows": 23786,
        "mean_overlap_at_N": 0.15337008894446777,
        "mean_de_sig_genes_recall": 0.45229413027682747,
        "mean_pearson_delta": 0.3581256233091921
      },
      {
        "feature": "protocol",
        "value": "fewshot",
        "n_rows": 16022,
        "mean_overlap_at_N": 0.3539402483266169,
        "mean_de_sig_genes_recall": 0.4567349167198684,
        "mean_pearson_delta": 0.643114361363374
      },
      {
        "feature": "representation",
        "value": "SE",
        "n_rows": 19873,
        "mean_overlap_at_N": 0.20521228255937513,
        "mean_de_sig_genes_recall": 0.4600542721940042,
        "mean_pearson_delta": 0.43311294942219586
      },
      {
        "feature": "representation",
        "value": "HVG",
        "n_rows": 19935,
        "mean_overlap_at_N": 0.2628897869586007,
        "mean_de_sig_genes_recall": 0.4481273423251553,
        "mean_pearson_delta": 0.5124204028055743
      }
    ]
  }
}
```

## All scored-evidence candidate screens

These are the actual original exploratory group comparisons. Candidates grouped here were not all independent completed SPC experiments. No extra confirmation or curve is invented for a rejected eligibility decision.

In [6]:
ns.display_json(rp.COMPUTED/'initial/artifacts/all_scored_axis_discovery.json')
with pd.option_context('display.max_rows',200):
    display(pd.read_csv(rp.COMPUTED/'continued/artifacts/all_scored_axis_discovery_continued.csv'))

```json
[
  {
    "candidate_axis": "dataset_family",
    "feature": "dataset_family",
    "metric": "overlap_at_N",
    "n_levels_ge20": 3,
    "mean_spread": 0.17503643377301334,
    "best_level": "Tahoe-100M",
    "worst_level": "Replogle-Nadig",
    "status": "outcome_informed_discovery_only",
    "level_summary": [
      {
        "dataset_family": "Replogle-Nadig",
        "count": 19634,
        "mean": 0.14595008628289993,
        "std": 0.18690592279660315
      },
      {
        "dataset_family": "Parse-PBMC",
        "count": 1458,
        "mean": 0.3057091158280087,
        "std": 0.22838419033865623
      },
      {
        "dataset_family": "Tahoe-100M",
        "count": 18716,
        "mean": 0.32098652005591327,
        "std": 0.2077607082802934
      }
    ]
  },
  {
    "candidate_axis": "tahoe_moa_fine",
    "feature": "drugmeta_moa-fine",
    "metric": "overlap_at_N",
    "n_levels_ge20": 26,
    "mean_spread": 0.1250947529686372,
    "best_level": "Proteasome inhibitor",
    "worst_level": "DNA methyltransferase inhibitor",
    "status": "outcome_informed_discovery_only",
    "level_summary": [
      {
        "drugmeta_moa-fine": "DNA methyltransferase inhibitor",
        "count": 170,
        "mean": 0.27616100745483946,
        "std": 0.22264305114801863
      },
      {
        "drugmeta_moa-fine": "EGFR/ERBB inhibitor",
        "count": 570,
        "mean": 0.2849476214461245,
        "std": 0.19551559364736273
      },
      {
        "drugmeta_moa-fine": "Other TK inhibitor",
        "count": 660,
        "mean": 0.297224189320638,
        "std": 0.20392905573083311
      },
      {
        "drugmeta_moa-fine": "Sonic inhibitor",
        "count": 140,
        "mean": 0.3003832503856378,
        "std": 0.21637585854878172
      },
      {
        "drugmeta_moa-fine": "Other MAPK inhibitor",
        "count": 510,
        "mean": 0.30071389129715925,
        "std": 0.1987891831449382
      },
      {
        "drugmeta_moa-fine": "Adrenoceptor agonist",
        "count": 380,
        "mean": 0.3017080075419724,
        "std": 0.20101672085190392
      },
      {
        "drugmeta_moa-fine": "GSK3 inhibitor",
        "count": 180,
        "mean": 0.30232215419776703,
        "std": 0.19773007118244657
      },
      {
        "drugmeta_moa-fine": "Multi-TK inhibitor",
        "count": 460,
        "mean": 0.3045801241017002,
        "std": 0.201523216862092
      },
      {
        "drugmeta_moa-fine": "PI3K/AKT inhibitor",
        "count": 270,
        "mean": 0.3097948316660418,
        "std": 0.1941315936210903
      },
      {
        "drugmeta_moa-fine": "unclear",
        "count": 9750,
        "mean": 0.3109168530328336,
        "std": 0.2101795915631771
      },
      {
        "drugmeta_moa-fine": "MEK inhibitor",
        "count": 240,
        "mean": 0.311338646650397,
        "std": 0.1611036517894177
      },
      {
        "drugmeta_moa-fine": "RAS inhibitor",
        "count": 210,
        "mean": 0.32506413749842294,
        "std": 0.17216278104725263
      },
      {
        "drugmeta_moa-fine": "Cyclooxygenase inhibitor",
        "count": 680,
        "mean": 0.32621346414191765,
        "std": 0.21332294822481107
      },
      {
        "drugmeta_moa-fine": "Androgen receptor antagonist",
        "count": 190,
        "mean": 0.326783988858525,
        "std": 0.215699808148212
      },
      {
        "drugmeta_moa-fine": "JAK/STAT inhibitor",
        "count": 340,
        "mean": 0.3389276617515626,
        "std": 0.19837678233757594
      },
      {
        "drugmeta_moa-fine": "DNA synthesis/repair inhibitor",
        "count": 1290,
        "mean": 0.3399195006501739,
        "std": 0.21772186120228457
      },
      {
        "drugmeta_moa-fine": "Glucose transporter inhibitor",
        "count": 220,
        "mean": 0.36091802247429944,
        "std": 0.22978329260779587
      },
      {
        "drugmeta_moa-fine": "MTOR inhibitor",
        "count": 420,
        "mean": 0.3627186511143625,
        "std": 0.19502971420153278
      },
      {
        "drugmeta_moa-fine": "RAF inhibitor",
        "count": 176,
        "mean": 0.3629422096549461,
        "std": 0.21862237525712688
      },
      {
        "drugmeta_moa-fine": "CDK inhibitor",
        "count": 300,
        "mean": 0.36453195823604145,
        "std": 0.18202063767048857
      },
      {
        "drugmeta_moa-fine": "Glucocorticoid receptor agonist",
        "count": 280,
        "mean": 0.36815421698819356,
        "std": 0.1749659406004725
      },
      {
        "drugmeta_moa-fine": "Protein synthesis inhibitor",
        "count": 330,
        "mean": 0.3687316954251153,
        "std": 0.20772718551985594
      },
      {
        "drugmeta_moa-fine": "Retinoic receptor agonist",
        "count": 220,
        "mean": 0.3786191287712921,
        "std": 0.18232001702212203
      },
      {
        "drugmeta_moa-fine": "HDAC inhibitor",
        "count": 170,
        "mean": 0.3923557948025106,
        "std": 0.2192747623462327
      },
      {
        "drugmeta_moa-fine": "Microtubule inhibitor",
        "count": 390,
        "mean": 0.3943236764800518,
        "std": 0.1943796990550051
      },
      {
        "drugmeta_moa-fine": "Proteasome inhibitor",
        "count": 170,
        "mean": 0.40125576042347666,
        "std": 0.1710101063432151
      }
    ]
  },
  {
    "candidate_axis": "representation",
    "feature": "representation",
    "metric": "overlap_at_N",
    "n_levels_ge20": 2,
    "mean_spread": 0.05767750439922559,
    "best_level": "HVG",
    "worst_level": "SE",
    "status": "outcome_informed_discovery_only",
    "level_summary": [
      {
        "representation": "SE",
        "count": 19873,
        "mean": 0.20521228255937513,
        "std": 0.21846144447015012
      },
      {
        "representation": "HVG",
        "count": 19935,
        "mean": 0.2628897869586007,
        "std": 0.21124004635238766
      }
    ]
  },
  {
    "candidate_axis": "tahoe_moa_broad",
    "feature": "drugmeta_moa-broad",
    "metric": "overlap_at_N",
    "n_levels_ge20": 3,
    "mean_spread": 0.020772902315989994,
    "best_level": "activator/agonist",
    "worst_level": "unclear",
    "status": "outcome_informed_discovery_only",
    "level_summary": [
      {
        "drugmeta_moa-broad": "unclear",
        "count": 3760,
        "mean": 0.31742390262078196,
        "std": 0.2144528791434508
      },
      {
        "drugmeta_moa-broad": "inhibitor/antagonist",
        "count": 13626,
        "mean": 0.32028974643669644,
        "std": 0.20629451918224606
      },
      {
        "drugmeta_moa-broad": "activator/agonist",
        "count": 1330,
        "mean": 0.33819680493677196,
        "std": 0.20279100072022097
      }
    ]
  },
  {
    "candidate_axis": "tahoe_dose",
    "feature": "dose",
    "metric": "overlap_at_N",
    "n_levels_ge20": 3,
    "mean_spread": 0.008059252214153267,
    "best_level": "0.5",
    "worst_level": "5.0",
    "status": "outcome_informed_discovery_only",
    "level_summary": [
      {
        "dose": 5.0,
        "count": 5780,
        "mean": 0.3171153501558342,
        "std": 0.2070889144932
      },
      {
        "dose": 0.05,
        "count": 6470,
        "mean": 0.3202593518611148,
        "std": 0.20882185886477744
      },
      {
        "dose": 0.5,
        "count": 6466,
        "mean": 0.32517460236998746,
        "std": 0.20725150445593277
      }
    ]
  }
]
```

,feature,value,n_rows,mean_overlap_at_N,mean_de_sig_genes_recall,mean_pearson_delta
0,chem_support_level,L3_mid,1458,0.505348,0.687487,0.905180
1,chem_support_level,L1_near,1458,0.513540,0.688316,0.927211
2,chem_support_level,L4_low_mid,1457,0.523762,0.694204,0.917136
3,chem_support_level,L5_far,1458,0.526985,0.686794,0.925106
4,chem_support_level,L2_high_mid,1457,0.532182,0.704359,0.921263
5,context,Plasmablast,304,0.038826,0.111781,0.194551
6,context,jurkat,5242,0.127856,0.159926,0.264831
7,context,rpe1,5120,0.149371,0.170018,0.323330
8,context,k562,4702,0.149490,0.212233,0.273690
9,context,hepg2,4570,0.159231,0.199895,0.318622


## Refined candidates and ontology checks — retained and discarded

In [7]:
display(pd.read_csv(rp.COMPUTED/'continued/artifacts/replogle_refined_axis_screen.csv'))
display(pd.read_csv(rp.COMPUTED/'continued/artifacts/multi_pathway_support_screen.csv'))
display(pd.DataFrame(json.loads((root/'axis_catalog.json').read_text())))

,axis_id,feature,status,discovery_pass,confirmation_pass,all_context_pass,score_confirmation_mean_high_minus_low,n_confirmation_representations,discovery_high_minus_low_mean,confirmation_high_minus_low_mean
0,STATE_REPLOGLE_REACTOME_PATHWAY_LOG_TRAIN_SUPP...,reactome_pathway_train_support_log_mean,tested,True,True,True,0.118563,2,0.147842,0.118563
1,STATE_REPLOGLE_REACTOME_PATHWAY_MEAN_TRAIN_SUP...,reactome_pathway_train_support_mean,tested,True,True,True,0.092002,2,0.119627,0.092002
2,STATE_REPLOGLE_REACTOME_PATHWAY_MIN_TRAIN_SUPP...,reactome_pathway_train_support_min,tested,True,True,True,0.115978,2,0.163731,0.115978
3,STATE_REPLOGLE_HGNC_GROUP_LOG_TRAIN_SUPPORT_V2,hgnc_group_train_support_log_mean,tested,True,True,True,0.104640,2,0.162565,0.104640
4,STATE_REPLOGLE_SYMBOL_FAMILY_SUPPORT_CONTROL,symbol_family_candidate_train_count,tested,True,True,True,0.094010,2,0.240916,0.094010


,collection,status,n_genes_with_features,n_unique_features,cutpoints_from_fewshot_discovery_norm_support,fewshot_confirmation_hvg_gap,fewshot_confirmation_se_gap,zeroshot_hvg_gap,zeroshot_se_gap,fewshot_support_coef_controlled,levels_artifact,trends_artifact
0,Reactome,supports_broader_claim,11461,2823,"[0.38495730076261436, 0.45794964463954363, 0.5...",0.114425,0.085313,0.067595,0.043669,"{'n_rows': 6640, 'coef': 0.028550118245131033,...",/private/tmp/spectra-pr-integration-20261108/r...,/private/tmp/spectra-pr-integration-20261108/r...
1,HGNC_gene_group,supports_broader_claim,27676,2037,"[0.1485535550192543, 0.23176698890230232, 0.34...",0.114409,0.093496,0.067848,0.047095,"{'n_rows': 6890, 'coef': 0.027792203160281075,...",/private/tmp/spectra-pr-integration-20261108/r...,/private/tmp/spectra-pr-integration-20261108/r...
2,GO_BP,supports_broader_claim,31552,11190,"[0.1373479432020825, 0.21412019957433295, 0.32...",0.161422,0.134550,0.087284,0.056794,"{'n_rows': 7894, 'coef': 0.0145135407973098, '...",/private/tmp/spectra-pr-integration-20261108/r...,/private/tmp/spectra-pr-integration-20261108/r...
3,GO_MF,supports_broader_claim,27010,4855,"[0.3754278233733169, 0.4969000292315334, 0.665...",0.014290,0.054462,0.034752,0.048388,"{'n_rows': 7978, 'coef': 0.0036247803343459895...",/private/tmp/spectra-pr-integration-20261108/r...,/private/tmp/spectra-pr-integration-20261108/r...
4,GO_CC,supports_broader_claim,32339,1944,"[0.45701104549789345, 0.5531692739379114, 0.64...",0.121489,0.090130,0.081189,0.054814,"{'n_rows': 8040, 'coef': 0.006538185810734801,...",/private/tmp/spectra-pr-integration-20261108/r...,/private/tmp/spectra-pr-integration-20261108/r...
5,WikiPathways,supports_broader_claim,8960,985,"[0.1836709931576028, 0.2634902745869932, 0.367...",0.103912,0.126574,0.034633,0.034359,"{'n_rows': 4602, 'coef': 0.018654204442852644,...",/private/tmp/spectra-pr-integration-20261108/r...,/private/tmp/spectra-pr-integration-20261108/r...
6,KEGG,supports_broader_claim,29104,371,"[0.2690731414998516, 0.34765048335345344, 0.42...",0.141454,0.127841,0.066405,0.046664,"{'n_rows': 4414, 'coef': 0.025459552357949382,...",/private/tmp/spectra-pr-integration-20261108/r...,/private/tmp/spectra-pr-integration-20261108/r...


,number,title,notebook,stage,curve,original_agent_determination
0,1,Chemical similarity to training drugs,notebooks/test-01.ipynb,initial,spc_tahoe_fewshot_target_context_chemical_supp...,No meaningful decline. HVG overlap@N was 0.013...
1,2,Dose represented in training,notebooks/test-02.ipynb,initial,exploratory_tahoe_dose_protocol_curve.csv,The agent plotted dose as an exploratory proto...
2,3,Reactome pathway-set similarity,notebooks/test-03.ipynb,replogle,spc_replogle_fewshot_reactome_support.csv,Above versus at or below 0.5 Jaccard similarit...
3,4,Reactome mean,notebooks/test-04.ipynb,replogle,spc_state_replogle_reactome_pathway_mean_train...,Passed the five-measure screen. High minus low...
4,5,Reactome log-mean (REP),notebooks/test-05.ipynb,replogle,spc_state_replogle_reactome_pathway_log_train_...,Passed the five-measure screen. High minus low...
5,6,Reactome minimum,notebooks/test-06.ipynb,replogle,spc_state_replogle_reactome_pathway_min_train_...,Passed the five-measure screen. High minus low...
6,7,HGNC log-mean,notebooks/test-07.ipynb,replogle,spc_state_replogle_hgnc_group_log_train_suppor...,Passed the five-measure screen. High minus low...
7,8,Gene-symbol prefix,notebooks/test-08.ipynb,replogle,spc_state_replogle_symbol_family_support_contr...,Passed the five-measure screen. High minus low...
8,9,Cell context and annotation counts,notebooks/test-09.ipynb,controls,controls_replogle_best_axis_level_summary_all.csv,The association persisted after excluding unan...
9,10,Multivariable adjustment,notebooks/test-10.ipynb,controls,None,The adjusted support coefficient remained posi...


## Original run narrative

In [8]:
display(Markdown((rp.CONTINUED/'state_spectra_report.md').read_text()))

# STATE SPECTRA Generalizability Audit

Run root: `/ewsc/yektefai/spectra_runs/state_generalizability_continued_20260603`

This run used official precomputed STATE result CSVs from the prior SPECTRA
session, not fresh STATE inference. The audit used 39,808 scored result rows
across Tahoe-100M, Replogle-Nadig, and Parse-PBMC, with HVG and SE
representations. Prospective metadata came from official split TOMLs, HGNC, and
Reactome.

## Primary Supported Axis

**Axis:** Replogle Reactome pathway train-support density.

For each held-out Replogle gene perturbation, the similarity score is the mean
`log(1 + number of same-context training genes sharing each Reactome pathway of
the target gene)`. Higher support means the target gene is closer to known
training biology in that target context.

Discovery contexts were `hepg2` and `k562`; confirmation contexts were `jurkat`
and `rpe1`. Split membership used only prospective gene/pathway metadata and
official train/test definitions, not target-model errors or reference-derived
outcomes.

### Few-Shot Confirmation

Held-out confirmation rows: 2,145 per representation.

| Representation | Low/no support mean overlap@N | High support mean overlap@N | High-low gap |
|---|---:|---:|---:|
| HVG | 0.2036 | 0.3313 | +0.1278 |
| SE | 0.1576 | 0.2670 | +0.1094 |

Excluding the no-annotation bucket, the confirmation high-low gaps were +0.1482
for HVG and +0.1280 for SE.

### Per-Context Control

High-low overlap@N gaps were positive in every tested context and
representation:

| Context | HVG gap | SE gap |
|---|---:|---:|
| hepg2 | +0.1949 | +0.1752 |
| k562 | +0.1275 | +0.0966 |
| jurkat | +0.1372 | +0.0734 |
| rpe1 | +0.1233 | +0.1198 |

### Cross-Protocol Zero-Shot Check

The same support idea was applied to Replogle zero-shot by using training genes
from the other cell contexts and normalizing support by train-set size. This
used 5,759 rows per representation.

| Representation | Low/no support mean overlap@N | High support mean overlap@N | High-low gap |
|---|---:|---:|---:|
| HVG | 0.1095 | 0.1771 | +0.0676 |
| SE | 0.0563 | 0.0999 | +0.0437 |

### Confound Checks

The trend remained after excluding the no-annotation bucket and within
pathway-count matched bins. A simple OLS control over context, representation,
subset, Reactome pathway count, and label-derived DE-count severity kept a
positive support coefficient for overlap@N (`+0.0284`, t=14.36). DE-count
severity was much larger (`+0.1629`), so perturbation strength is an important
outcome confound/mediator, but it did not fully explain the support axis.

## Negative / Non-Terminal Axes

- Prior Tahoe target-context chemical Tanimoto support was split-valid but
  negative/weak: no meaningful degradation with lower chemical support.
- Parse-PBMC ligand-family training support reversed direction: higher
  same-family support did not improve overlap@N in discovery or confirmation.

## Current Interpretation

STATE appears to generalize worse for Replogle gene perturbations whose Reactome
pathways are sparsely represented among training perturbations. This boundary
holds across held-out few-shot contexts and also weakly but consistently in the
harder Replogle zero-shot protocol.

This is a current-pool confirmed SPECTRA axis, not yet a raw-data rerun. The
main remaining reviewer-facing gap is a fixed baseline or raw-label rerun, which
requires raw single-cell matrices/predictions rather than only official scored
CSV outputs.

## Cross-Ontology Extension

To test whether this was Reactome-specific, the same SPECTRA protocol was
repeated with independent prospective gene-set systems: HGNC gene groups, GO
Biological Process, GO Molecular Function, GO Cellular Component, WikiPathways,
and KEGG. For each collection, support levels were frozen from `hepg2+k562`
few-shot discovery contexts, then evaluated on `jurkat+rpe1` few-shot
confirmation and Replogle zero-shot cross-protocol confirmation.

| Collection | Few-shot HVG gap | Few-shot SE gap | Zero-shot HVG gap | Zero-shot SE gap |
|---|---:|---:|---:|---:|
| Reactome | +0.1144 | +0.0853 | +0.0676 | +0.0437 |
| HGNC gene group | +0.1144 | +0.0935 | +0.0678 | +0.0471 |
| GO Biological Process | +0.1614 | +0.1345 | +0.0873 | +0.0568 |
| GO Molecular Function | +0.0143 | +0.0545 | +0.0348 | +0.0484 |
| GO Cellular Component | +0.1215 | +0.0901 | +0.0812 | +0.0548 |
| WikiPathways | +0.1039 | +0.1266 | +0.0346 | +0.0344 |
| KEGG | +0.1415 | +0.1278 | +0.0664 | +0.0467 |

This supports the broader claim: the degradation is not Reactome-specific. It
is a pathway/module train-coverage effect. The strongest independent support
came from GO Biological Process, KEGG, WikiPathways, Reactome, HGNC groups, and
GO Cellular Component. GO Molecular Function had positive high-low gaps, but its
controlled few-shot coefficient was weak compared with the other systems.

High-support bins were dominated by recurrent biological modules such as
translation, ribosome, RNA metabolism, mRNA splicing, rRNA processing,
mitochondrial translation, cytoplasmic ribosomal proteins, and spliceosome
pathways.

## Key Artifacts

- `artifacts/replogle_refined_axis_screen.csv`
- `artifacts/spc_replogle_best_refined_axis.csv`
- `artifacts/spc_replogle_best_refined_axis_trends.csv`
- `artifacts/spc_replogle_best_refined_axis_context_trends.csv`
- `artifacts/controls_replogle_best_axis_high_low_gaps.csv`
- `artifacts/controls_replogle_best_axis_pathway_count_matched.csv`
- `artifacts/controls_replogle_best_axis_regression.json`
- `artifacts/spc_replogle_zeroshot_reactome_support_confirmation.csv`
- `artifacts/spc_replogle_zeroshot_reactome_support_confirmation_trends.csv`
- `artifacts/multi_pathway_support_screen.csv`
- `artifacts/multi_pathway_support_summary.json`
- `artifacts/multi_pathway_top_features_by_support_level.json`
- `spectra_ready_dataset/state_replogle_reactome_pathway_log_train_support_v2_examples.csv`
- `spectra_ready_dataset/replogle_zeroshot_reactome_support_confirmation_examples.csv`


### Replay detail: tied nearest-reference labels

All reconstructed similarities, frozen levels and checked outcome metrics match the original. In 50 records, the displayed name of the nearest reference drug differs because two reference drugs have exactly the same maximum Tanimoto similarity and the original code iterates an unordered Python set. Both pairwise similarities were checked. This does not change the curve. Per-record evidence is in `computed/initial/nearest_reference_ties.json`.

`download_manifest.json` lists all 58 score/metadata files with the recovered revision, SHA-256, byte size and a pinned download URL derived from that revision. Historical annotation snapshots are included under `original/`; current catalog URLs are not substitutes for their archived bytes.